# 第13章: 深層学習によるSLAM - 学習された特徴量と深度推定

本ノートブックでは、深層学習がSLAMにもたらす革新について学びます。

## 学習内容
1. **学習された特徴記述子**: 従来の手作り特徴量（ORB等）と学習ベースの特徴量の比較
2. **最近傍マッチング**: 記述子ベクトルによる特徴点対応付け
3. **単眼深度推定**: スケール曖昧性の問題とその影響
4. **古典的手法 vs 学習ベース手法**: それぞれの長所と短所

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import ConnectionPatch
np.random.seed(42)

## 1. 特徴記述子とは

SLAMにおける特徴点マッチングでは、画像中の各キーポイントに対して「記述子（descriptor）」と呼ばれるベクトルを計算します。

- **古典的手法（ORB, SIFT等）**: 手作りの勾配ヒストグラムやバイナリパターンで記述子を生成
- **学習ベース手法（SuperPoint, D2-Net等）**: CNNで画像から直接、識別力の高い記述子を学習

学習された特徴量は、照明変化・視点変化・テクスチャレスな領域に対してよりロバストです。

ここでは、記述子ベクトルのシミュレーションを通じて、マッチングの仕組みを理解します。

In [ ]:
# --- Simulate learned feature descriptors and nearest-neighbor matching ---

def generate_keypoints(n_points, image_size=(640, 480)):
    """Generate random keypoint locations."""
    x = np.random.uniform(20, image_size[0] - 20, n_points)
    y = np.random.uniform(20, image_size[1] - 20, n_points)
    return np.column_stack([x, y])

def generate_descriptors(n_points, dim=128, noise_std=0.0):
    """Generate unit-norm descriptor vectors (simulating CNN output)."""
    desc = np.random.randn(n_points, dim)
    desc = desc / np.linalg.norm(desc, axis=1, keepdims=True)
    if noise_std > 0:
        desc += np.random.randn(n_points, dim) * noise_std
        desc = desc / np.linalg.norm(desc, axis=1, keepdims=True)
    return desc

def nearest_neighbor_match(desc1, desc2, ratio_threshold=0.8):
    """Match descriptors using nearest neighbor with Lowe's ratio test."""
    # Compute cosine similarity (since descriptors are unit-norm, dot product = cosine sim)
    similarity = desc1 @ desc2.T  # (N, M)
    
    matches = []
    for i in range(len(desc1)):
        sorted_idx = np.argsort(-similarity[i])  # descending
        best = similarity[i, sorted_idx[0]]
        second_best = similarity[i, sorted_idx[1]]
        # Ratio test: best match must be significantly better than second best
        if second_best > 0 and best / second_best > 1.0 / ratio_threshold:
            matches.append((i, sorted_idx[0], best))
    return matches

# Generate two sets of keypoints (simulating two camera views)
n_points = 30
kp1 = generate_keypoints(n_points)
kp2 = kp1 + np.random.randn(n_points, 2) * 15 + np.array([30, 5])  # shifted + noise

# Classical descriptors: higher noise, lower discrimination
desc1_classical = generate_descriptors(n_points, dim=32)
desc2_classical = desc1_classical.copy()
desc2_classical += np.random.randn(n_points, 32) * 0.6  # significant noise
desc2_classical /= np.linalg.norm(desc2_classical, axis=1, keepdims=True)

# Learned descriptors: lower noise, better discrimination
desc1_learned = generate_descriptors(n_points, dim=128)
desc2_learned = desc1_learned.copy()
desc2_learned += np.random.randn(n_points, 128) * 0.15  # less noise
desc2_learned /= np.linalg.norm(desc2_learned, axis=1, keepdims=True)

# Match both
matches_classical = nearest_neighbor_match(desc1_classical, desc2_classical)
matches_learned = nearest_neighbor_match(desc1_learned, desc2_learned)

# Count correct matches (ground truth: index i should match index i)
correct_classical = sum(1 for i, j, _ in matches_classical if i == j)
correct_learned = sum(1 for i, j, _ in matches_learned if i == j)

print(f"Classical descriptors (dim=32, high noise):")
print(f"  Matches found: {len(matches_classical)}, Correct: {correct_classical}")
print(f"\nLearned descriptors (dim=128, low noise):")
print(f"  Matches found: {len(matches_learned)}, Correct: {correct_learned}")

In [ ]:
# --- Visualize feature matching results ---

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, matches, title, kp_1, kp_2 in [
    (axes[0], matches_classical, f"Classical Features\n(Matches: {len(matches_classical)}, Correct: {correct_classical})", kp1, kp2),
    (axes[1], matches_learned, f"Learned Features\n(Matches: {len(matches_learned)}, Correct: {correct_learned})", kp1, kp2),
]:
    ax.set_xlim(-50, 700)
    ax.set_ylim(-50, 530)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    # Draw image frames
    ax.add_patch(plt.Rectangle((0, 0), 640, 480, fill=False, edgecolor='gray', linewidth=2, linestyle='--'))
    ax.text(320, -20, 'Image 1 & 2 (overlaid)', ha='center', fontsize=9, color='gray')
    
    # Draw keypoints
    ax.scatter(kp_1[:, 0], kp_1[:, 1], c='blue', s=40, zorder=5, label='Image 1')
    ax.scatter(kp_2[:, 0], kp_2[:, 1], c='red', s=40, zorder=5, label='Image 2')
    
    # Draw matches
    for i, j, score in matches:
        color = 'green' if i == j else 'red'
        alpha = 0.8 if i == j else 0.4
        ax.plot([kp_1[i, 0], kp_2[j, 0]], [kp_1[i, 1], kp_2[j, 1]], 
                color=color, alpha=alpha, linewidth=1.5)
    
    ax.legend(loc='lower right', fontsize=9)
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig('feature_matching_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("緑線 = 正しいマッチ、赤線 = 誤マッチ")

## 2. 単眼深度推定とスケール曖昧性

単眼カメラからの深度推定は、深層学習の重要な応用の一つです。しかし、単眼深度推定には本質的な**スケール曖昧性（scale ambiguity）**が存在します。

- 同じ画像は、小さな物体が近くにある場合と、大きな物体が遠くにある場合で全く同じに見える
- CNNは相対的な深度（近い/遠い）は推定できるが、絶対スケールは決定できない
- SLAMで使用する際は、スケールの整合性を保つ追加的な仕組みが必要

In [ ]:
# --- Monocular depth prediction: scale ambiguity demonstration ---

def simulate_depth_map(shape=(60, 80)):
    """Create a synthetic scene depth map with objects at various depths."""
    depth = np.ones(shape) * 10.0  # background at 10m
    # Near object (box)
    depth[15:30, 10:25] = 3.0
    # Mid object (cylinder-like)
    cy, cx, r = 35, 55, 10
    yy, xx = np.ogrid[:shape[0], :shape[1]]
    mask = (yy - cy)**2 + (xx - cx)**2 < r**2
    depth[mask] = 6.0
    # Far object
    depth[5:15, 50:70] = 8.0
    # Add smooth gradient noise
    depth += np.random.randn(*shape) * 0.2
    return depth

# Ground truth depth
gt_depth = simulate_depth_map()

# Simulated CNN prediction (relative depth, unknown scale)
scale_factor = 0.35  # CNN predicts up-to-scale
predicted_relative = gt_depth * scale_factor + np.random.randn(*gt_depth.shape) * 0.15

# Show scale ambiguity
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

im0 = axes[0].imshow(gt_depth, cmap='plasma')
axes[0].set_title('Ground Truth Depth [m]', fontsize=12, fontweight='bold')
plt.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(predicted_relative, cmap='plasma')
axes[1].set_title('CNN Prediction (unknown scale)', fontsize=12, fontweight='bold')
plt.colorbar(im1, ax=axes[1], shrink=0.8)

# Align scale using median
scale_est = np.median(gt_depth) / np.median(predicted_relative)
aligned = predicted_relative * scale_est
im2 = axes[2].imshow(aligned, cmap='plasma')
axes[2].set_title(f'Scale-Aligned (factor={scale_est:.2f})', fontsize=12, fontweight='bold')
plt.colorbar(im2, ax=axes[2], shrink=0.8)

for ax in axes:
    ax.set_xlabel('x')
    ax.set_ylabel('y')

plt.tight_layout()
plt.savefig('depth_scale_ambiguity.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"スケール推定値: {scale_est:.3f} (真値: {1/scale_factor:.3f})")
print(f"整合後の平均誤差: {np.mean(np.abs(aligned - gt_depth)):.3f} m")

## 3. 古典的特徴量 vs 学習ベース特徴量の比較

以下では、ノイズ耐性と記述子の識別力を定量的に比較します。記述子間の類似度分布を見ることで、学習ベース手法がなぜ優れたマッチング性能を発揮するか理解できます。

In [ ]:
# --- Comparison: descriptor discriminability under noise ---

noise_levels = np.linspace(0.0, 1.0, 20)
n_test = 50
dims_classical = 32
dims_learned = 128

precision_classical = []
precision_learned = []

for noise in noise_levels:
    # Classical
    d1 = generate_descriptors(n_test, dim=dims_classical)
    d2 = d1 + np.random.randn(n_test, dims_classical) * noise
    d2 /= np.linalg.norm(d2, axis=1, keepdims=True)
    matches = nearest_neighbor_match(d1, d2, ratio_threshold=0.8)
    correct = sum(1 for i, j, _ in matches if i == j)
    precision_classical.append(correct / max(len(matches), 1))
    
    # Learned
    d1 = generate_descriptors(n_test, dim=dims_learned)
    d2 = d1 + np.random.randn(n_test, dims_learned) * noise
    d2 /= np.linalg.norm(d2, axis=1, keepdims=True)
    matches = nearest_neighbor_match(d1, d2, ratio_threshold=0.8)
    correct = sum(1 for i, j, _ in matches if i == j)
    precision_learned.append(correct / max(len(matches), 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Precision vs noise
axes[0].plot(noise_levels, precision_classical, 'o-', label=f'Classical (dim={dims_classical})', color='orange')
axes[0].plot(noise_levels, precision_learned, 's-', label=f'Learned (dim={dims_learned})', color='blue')
axes[0].set_xlabel('Noise Level (std)', fontsize=12)
axes[0].set_ylabel('Matching Precision', fontsize=12)
axes[0].set_title('Noise Robustness Comparison', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-0.05, 1.05)

# Similarity distribution
d_base = generate_descriptors(100, dim=128)
correct_sim = np.array([d_base[i] @ (d_base[i] + np.random.randn(128)*0.2) / 
                         np.linalg.norm(d_base[i] + np.random.randn(128)*0.2) for i in range(100)])
wrong_sim = np.array([d_base[i] @ d_base[(i+1) % 100] for i in range(100)])

axes[1].hist(correct_sim, bins=25, alpha=0.7, label='Correct pairs', color='green', density=True)
axes[1].hist(wrong_sim, bins=25, alpha=0.7, label='Incorrect pairs', color='red', density=True)
axes[1].set_xlabel('Cosine Similarity', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('Descriptor Similarity Distribution', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('descriptor_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## 演習問題

1. **記述子の次元数の影響**: `dims_learned` を 64, 128, 256 に変えて、ノイズ耐性がどう変化するか確認してください。次元数を増やすと常に良くなりますか？
2. **ratio test の閾値**: `ratio_threshold` を 0.6, 0.7, 0.8, 0.9 と変えて、precision と recall のトレードオフを観察してください。
3. **スケール推定**: 深度マップのスケール整合で、中央値の代わりに最小二乗法を使った場合の精度を比較してください。

## まとめ

- **学習された特徴記述子**は、高次元空間での表現力とノイズ耐性において古典的手法を上回る
- **最近傍マッチング + ratio test** は特徴点対応の標準的な手法であり、学習ベース記述子と組み合わせることで高精度なマッチングが可能
- **単眼深度推定**は有用だが、**スケール曖昧性**が本質的に存在し、追加情報（IMU、既知のスケール等）による補正が必要
- 実際のシステム（SuperPoint + SuperGlue, LoFTR等）では、検出と記述子の統合学習やアテンション機構による高度なマッチングが行われている